In [1]:
import time, os
from dotenv import load_dotenv
from genai.credentials import Credentials
from genai.model import Model
from genai.schemas import GenerateParams
import numpy as np

In [3]:
model_id = 'google/flan-t5-xl'

In [4]:
load_dotenv()
api_key = os.getenv("GENAI_KEY", None) 
api_url = os.getenv("GENAI_API", None)
creds = Credentials(api_key, api_endpoint=api_url)
params = GenerateParams(decoding_method="greedy")
model = Model(model_id, params=params, credentials=creds)

In [5]:
dataset = {}
fnames = ['train', 'val', 'test']
for fname in fnames:
    data = {'label': [], 'question': []}
    with open(f'data/{fname}', 'r') as f:
        for line in f:
            label = int(line.split(' ')[0])
            question = line[2:]
            data['label'].append(label)
            data['question'].append(question)
    dataset[fname] = data

In [33]:
n_examples = 3
context = ''
for i in range(n_examples):
    relevant = 'Yes' if dataset['train']['label'][i] == 'True' else 'No'
    context += f"Question:{dataset['train']['question'][i]}. Relevant: {relevant}\n"

In [34]:
context

"Question:certainly , many algorithms are built complex and efficient by combining many algorithms into one ( as one step may also be considered an algorithm , is n't it ?\n. Relevant: No\nQuestion:but what i would like to ask - what are the algorithms considered to implement the biggest amount of advanced techniques in computational and/or natural sciences ?\n. Relevant: No\nQuestion:is it just based on the length of code ?\n. Relevant: No\n"

## Validation and Test set evaluation

In [69]:
def evaluate(name_set):
    prompts = []
    for i in range(len(dataset[name_set]['question'])):
        question = dataset[name_set]['question'][i]
        prompt = f'{context}Question:{question}. Relevant:'
        prompts.append(prompt)
    preds = model.generate(prompts)
    preds = [item.generated_text for item in preds]
    print('answers that are not "Yes" or "No":')
    for item in preds:
        if item not in ['Yes', 'No']:
            print(item)
    labels = np.array(dataset[name_set]['label']).astype(str)
    labels[labels == '1'] = 'Yes'
    labels[labels == '0'] = 'No'
    accuracy = (val_preds == labels).sum() / len(labels)
    print(f'{name_set} accuracy:{accuracy}')

In [70]:
evaluate('val')
evaluate('test')

answers that are not "Yes" or "No":
val accuracy:0.445
answers that are not "Yes" or "No":
test accuracy:0.475
